In [16]:
from pathlib import Path

import polars as pl
import yaml


with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [ ]:
user_pings = pl.DataFrame()
for i in range(256):
    data = pl.read_parquet(
        f"/mnt/common-hdd/group-dirs/tkom-2026-project/partitioned/{i:02x}/pings.parquet"
    )
    data = data.group_by(pl.col.device_id).agg(
        pl.col.timestamp.count().alias("number_of_pings")
    )
    user_pings = pl.concat([user_pings, data])
    del data
user_pings = user_pings.sort("device_id")

In [18]:
path = f"{config['dirs']['output']}/{config['city']}/descriptives"
Path(path).mkdir(exist_ok=True, parents=True)
user_pings.write_parquet(f"{path}/ping_counts_by_device.parquet")
user_pings.write_csv(f"{path}/ping_counts_by_device.csv")